# Notebook 1 — Crawl & Scrape Berita DetikFinance (L0 → L2)

Notebook ini **berdiri sendiri** (tidak mengimpor modul `.py` apa pun). Semua fungsi
ditulis langsung di sini supaya kamu bisa membaca, memahami, lalu menulis ulang dengan
gayamu sendiri.

## Apa yang dilakukan notebook ini

Crawl berita DetikFinance secara **bertingkat** (standar text-mining keuangan):

| Level | Isi | Alasan |
|-------|-----|--------|
| **L0** | metadata/link **semua** artikel per hari (judul, url, subkanal) | dokumentasi populasi lengkap |
| **L1** | seleksi **kandidat relevan** (judul memuat alias bank / kata kebijakan / subkanal bursa) + 2% sampel non-kandidat (kontrol negatif) | efisiensi; hanya ambil teks yang mungkin relevan |
| **L2** | unduh **HTML lengkap** kandidat (disimpan `.gz`) lalu parse jadi teks | reprodusibilitas: bisa re-parse tanpa crawl ulang |

## Output (sama dengan versi lama)
```
data/raw/html_index/YYYY/MM/DD/page_n.html.gz   <- HTML halaman indeks
data/raw/html_article/{id[:3]}/{id}.html.gz     <- HTML artikel (1 file = 1 artikel)
data/interim/news_links.parquet (+ .xlsx)        <- tabel semua link + flag kandidat
data/interim/news_clean_raw.parquet              <- teks ter-parse dari HTML kandidat
data/interim/crawl_checkpoint.json               <- progres (bisa stop & lanjut)
```

## Library yang dibutuhkan
```bash
pip install requests tenacity beautifulsoup4 lxml pandas pyarrow openpyxl tqdm
```

## 0. Import & konfigurasi

Semua pengaturan dikumpulkan di satu tempat agar mudah diubah. Perhatikan **etika crawling**:
1 request/detik + jeda acak (jitter), supaya tidak membebani server Detik.

In [ ]:
import gzip, hashlib, json, random, re, time
from datetime import date, datetime, timedelta
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential
from tqdm.auto import tqdm

# ----- Konfigurasi utama -----
BASE_URL      = "https://finance.detik.com/indeks"   # halaman indeks per tanggal
DATE_START    = date(2024, 1, 2)                       # awal rentang (Opsi A)
DATE_END      = date(2026, 9, 30)                      # akhir rentang
USER_AGENT    = "Mozilla/5.0 (research crawler; sentiment-topic study; contact: GANTI@email.anda)"
REQUEST_DELAY = 1.0        # jeda dasar antar-request (detik). PRODUKSI = 1.0 (sopan)
JITTER        = (0.5, 1.5) # jeda acak tambahan supaya pola tidak kaku
MAX_RETRIES   = 4          # berapa kali ulang bila server error
TIMEOUT       = 30         # batas tunggu per request (detik)
SEED          = 42         # agar sampel kontrol-negatif reprodusibel

# ----- Folder output -----
RAW     = Path("data/raw")
INTERIM = Path("data/interim")
RAW.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
print("Rentang crawl:", DATE_START, "s/d", DATE_END)

## 1. Kamus alias bank & kata kunci (untuk seleksi kandidat L1)

Ini daftar pola **regex** untuk mengenali apakah sebuah judul berita menyebut bank target,
grup bank BUMN, kata kunci kebijakan, atau kata pasar/makro. `\b` artinya "batas kata"
(supaya "BCA" tidak cocok dengan kata lain yang kebetulan mengandung "bca").

In [ ]:
BANK_ALIASES = {
    "BBCA": [r"\bBBCA\b", r"\bBCA\b", r"\bBank Central Asia\b", r"\bBank BCA\b"],
    "BBRI": [r"\bBBRI\b", r"\bBRI\b", r"\bBank Rakyat Indonesia\b", r"\bBank BRI\b"],
    "BBNI": [r"\bBBNI\b", r"\bBNI\b", r"\bBank Negara Indonesia\b", r"\bBank BNI\b"],
    "BMRI": [r"\bBMRI\b", r"\bBank Mandiri\b", r"\bMandiri\b"],
}
GROUP_ALIASES  = [r"\bHimbara\b", r"\bbank(?:-bank)? BUMN\b", r"\bbank besar\b", r"\bperbankan\b"]
POLICY_KEYWORDS = [r"\bOJK\b", r"\bBI Rate\b", r"\bsuku bunga\b", r"\bLPS\b", r"\bKUR\b",
                   r"\bNPL\b", r"\bkredit\b", r"\bdividen\b", r"\blaba bank\b"]
MARKET_KEYWORDS = [r"\bIHSG\b", r"\brupiah\b", r"\binflasi\b", r"\bditutup\b", r"\bdibuka\b"]
# subkanal yang otomatis dianggap kandidat (ringkasan bursa/pasar)
BURSA_SUBCHANNELS = ("bursa-dan-valas", "market-research")
NEG_SAMPLE_RATE   = 0.02   # ambil 2% non-kandidat sbg kontrol negatif (ukur recall)

# gabung semua pola kandidat jadi satu daftar regex terkompilasi (lebih cepat)
_ALL_PATTERNS = [re.compile(p, re.IGNORECASE)
                 for p in (sum(BANK_ALIASES.values(), []) + GROUP_ALIASES
                           + POLICY_KEYWORDS + MARKET_KEYWORDS)]

def is_candidate(title, subchannel):
    """True jika judul/subkanal menandakan berita MUNGKIN relevan (kandidat L1)."""
    if subchannel in BURSA_SUBCHANNELS:
        return True
    return any(p.search(title or "") for p in _ALL_PATTERNS)

## 2. Fungsi pengambil halaman (dengan sopan santun + retry)

`@retry` dari pustaka **tenacity** otomatis mengulang request bila server mengembalikan
error 429 (terlalu banyak request) atau 5xx, dengan jeda yang makin lama (exponential backoff).
Setelah tiap request berhasil kita `sleep` sebentar — ini kunci etika crawling.

In [ ]:
class RetryableHTTP(Exception):
    """Error yang layak diulang (429 / 503 / 5xx)."""

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

@retry(retry=retry_if_exception_type(RetryableHTTP),
       stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=2), reraise=True)
def fetch(url):
    """Ambil satu URL. Ulang otomatis bila server sibuk; jeda sopan setelah sukses."""
    resp = session.get(url, timeout=TIMEOUT)
    if resp.status_code in (429, 503) or resp.status_code >= 500:
        raise RetryableHTTP(f"{resp.status_code} pada {url}")
    resp.raise_for_status()
    # jeda = delay dasar + jitter acak (etika: jangan membanjiri server)
    time.sleep(REQUEST_DELAY + random.uniform(*JITTER))
    return resp.text

## 3. Fungsi penyimpanan HTML (gzip) & penamaan file

HTML disimpan terkompresi `.gz` (hemat ~70% ruang). Nama file artikel = **article_id**
(angka setelah `d-` pada URL, mis. `.../d-8688622/...` -> id `8688622`). File di-shard ke
subfolder 3 digit awal id supaya satu folder tidak berisi puluhan ribu file.

In [ ]:
def article_id_from_url(url):
    """Ambil id dari pola /d-<angka>/. Kalau tak ada, pakai hash URL."""
    m = re.search(r"/d-(\d+)/", url)
    return m.group(1) if m else hashlib.sha1(url.encode()).hexdigest()[:16]

def save_gz(html, path):
    """Simpan string HTML sebagai file gzip. Kembalikan ukuran byte terkompresi."""
    path.parent.mkdir(parents=True, exist_ok=True)
    data = gzip.compress(html.encode("utf-8"))
    path.write_bytes(data)
    return len(data)

def load_gz(path):
    """Baca kembali file gzip jadi string HTML (dipakai saat re-parse offline)."""
    return gzip.decompress(path.read_bytes()).decode("utf-8")

def index_path(day, page):
    return RAW / "html_index" / f"{day:%Y}/{day:%m}/{day:%d}" / f"page_{page}.html.gz"

def article_path(aid):
    return RAW / "html_article" / aid[:3] / f"{aid}.html.gz"

def article_relpath(aid):
    """Path relatif utk disimpan di tabel link (jembatan link <-> file HTML)."""
    return f"html_article/{aid[:3]}/{aid}.html.gz"

## 4. Parser: ambil field dari HTML indeks & artikel

Selektor CSS di bawah ini **terverifikasi** pada struktur DetikFinance (2024-2026):
- indeks: tiap artikel dalam `.list-content__item`, judul di `.media__title`, dll.
- artikel: judul `h1.detail__title`, waktu terbit di `<meta name="publishdate">`
  (format `YYYY/MM/DD HH:MM:SS`), isi di `.detail__body-text`.

Sebelum mengambil teks isi, elemen non-berita (script, caption foto, embed video, dll.)
dibuang dulu agar teks bersih.

In [ ]:
def subchannel_from_url(url):
    """Subkanal = segmen path pertama, mis. .../moneter/d-123/... -> 'moneter'."""
    m = re.search(r"finance\.detik\.com/([a-z0-9-]+)/d-\d+", url)
    return m.group(1) if m else None

def parse_index(html):
    """Dari HTML satu halaman indeks -> daftar dict artikel (url, judul, subkanal, tanggal)."""
    soup = BeautifulSoup(html, "lxml")
    items, seen = [], set()
    for node in soup.select(".list-content__item"):
        a = node.select_one("a.media__link") or node.select_one("a")
        if not a or not a.has_attr("href"):
            continue
        url = a["href"].split("?")[0].strip()
        if not re.search(r"/d-\d+/", url) or url in seen:
            continue
        seen.add(url)
        title = node.select_one(".media__title")
        datent = node.select_one(".media__date")
        items.append({
            "article_id": article_id_from_url(url),
            "url": url,
            "title": title.get_text(strip=True) if title else "",
            "subchannel": subchannel_from_url(url),
            "date_text": datent.get_text(" ", strip=True) if datent else None,
        })
    return items

# elemen yang dibuang dari isi artikel sebelum ekstraksi teks
_BODY_JUNK = ["script", "table", ".para_caption", ".lihatjg", ".lihatjuga",
              ".detail__long-nav", ".sisip_embed"]

def parse_article(html, url):
    """Dari HTML artikel -> dict (judul, waktu terbit, subkanal, teks isi, jumlah kata)."""
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.select_one("h1.detail__title")
    title = h1.get_text(strip=True) if h1 else ""

    # waktu terbit PERTAMA (bukan waktu 'diperbarui')
    published = None
    meta = soup.select_one('meta[name="publishdate"]')
    if meta and meta.has_attr("content"):
        for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S"):
            try:
                published = datetime.strptime(meta["content"].strip(), fmt).isoformat()
                break
            except ValueError:
                pass

    body = soup.select_one(".detail__body-text")
    text = ""
    if body:
        for sel in _BODY_JUNK:
            for junk in body.select(sel):
                junk.decompose()
        paras = [p.get_text(" ", strip=True) for p in body.select("p")]
        text = " ".join(p for p in paras if p)

    return {
        "article_id": article_id_from_url(url), "url": url, "title": title,
        "published_at": published, "subchannel": subchannel_from_url(url),
        "n_words": len(text.split()), "body_text": text,
    }

## 5. Checkpoint: supaya bisa stop & lanjut

Crawl penuh butuh berjam-jam. Kalau wifi putus / laptop dimatikan, kita tidak mau mengulang
dari awal. Checkpoint mencatat tanggal mana yang sudah selesai; saat dijalankan ulang,
tanggal itu dilewati. **Jangan hapus file `crawl_checkpoint.json`** kalau ingin melanjutkan.

In [ ]:
CKPT_PATH  = INTERIM / "crawl_checkpoint.json"
LINKS_PATH = INTERIM / "news_links.parquet"
CLEAN_PATH = INTERIM / "news_clean_raw.parquet"

def load_checkpoint():
    if CKPT_PATH.exists():
        return set(json.loads(CKPT_PATH.read_text()).get("done_days", []))
    return set()

def save_checkpoint(done_days):
    CKPT_PATH.write_text(json.dumps({"done_days": sorted(done_days)}))

def daterange(start, end):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

## 6. Proses utama crawl (L0 → L1 → L2, per hari)

Untuk **tiap tanggal**:
1. **L0** — ambil semua halaman indeks (berhenti saat halaman kosong), simpan HTML indeks,
   kumpulkan semua link + metadata.
2. **L1** — tandai tiap artikel kandidat/bukan (plus 2% non-kandidat sebagai kontrol negatif).
3. **L2** — untuk kandidat: unduh HTML lengkap, simpan `.gz`, parse jadi teks.
4. Tandai tanggal selesai di checkpoint, lalu simpan tabel (aman bila terputus).

In [ ]:
def run_crawl(start, end):
    done = load_checkpoint()
    # muat hasil sebelumnya agar bisa dilanjutkan (append, bukan menimpa)
    link_rows  = pd.read_parquet(LINKS_PATH).to_dict("records") if LINKS_PATH.exists() else []
    clean_rows = pd.read_parquet(CLEAN_PATH).to_dict("records") if CLEAN_PATH.exists() else []

    days = [d for d in daterange(start, end) if d.isoformat() not in done]
    for day in tqdm(days, desc="crawl per hari"):
        # ---- L0: semua halaman indeks hari ini ----
        day_items = []
        for page in range(1, 50):              # batas aman; biasanya ~6 halaman
            html = fetch(f"{BASE_URL}?page={page}&date={day:%m/%d/%Y}")
            if page == 1 or not day_items:
                save_gz(html, index_path(day, page))
            items = parse_index(html)
            if not items:                       # halaman kosong -> tanggal ini selesai
                break
            day_items.extend(items)

        # ---- L1 + L2: seleksi kandidat, lalu unduh HTML kandidat ----
        for it in day_items:
            cand = is_candidate(it["title"], it["subchannel"])
            neg  = False
            if not cand and random.random() < NEG_SAMPLE_RATE:
                cand, neg = True, True          # kontrol negatif
            row = {**it, "index_date": day.isoformat(),
                   "html_path": article_relpath(it["article_id"]),
                   "is_candidate": cand, "neg_control": neg, "html_saved": False}
            if cand:
                try:
                    ahtml = fetch(it["url"] + ("&" if "?" in it["url"] else "?") + "single=1")
                    save_gz(ahtml, article_path(it["article_id"]))
                    art = parse_article(ahtml, it["url"])
                    row["html_saved"] = True
                    clean_rows.append({**art, "index_date": day.isoformat()})
                except Exception as e:
                    print("  [WARN] gagal", it["url"], "->", e)
            link_rows.append(row)

        # ---- checkpoint + simpan (aman bila proses terputus) ----
        done.add(day.isoformat())
        save_checkpoint(done)
        pd.DataFrame(link_rows).to_parquet(LINKS_PATH, index=False)
        if clean_rows:
            pd.DataFrame(clean_rows).to_parquet(CLEAN_PATH, index=False)

    # ekspor Excel sekali di akhir utk kemudahan dibuka
    pd.DataFrame(link_rows).to_excel(INTERIM / "news_links.xlsx", index=False)
    return pd.DataFrame(link_rows), pd.DataFrame(clean_rows)

## 7. Jalankan — UJI dulu 1 minggu

**Selalu uji rentang kecil dulu** untuk memastikan output benar sebelum crawl penuh
(yang bisa makan 6-11 jam). Ganti dulu email kontak di `USER_AGENT` (sel 0).

In [ ]:
links, clean = run_crawl(DATE_START, date(2024, 1, 8))   # uji: 2-8 Jan 2024
print("link:", len(links), "| kandidat:", int(links.is_candidate.sum()),
      "| html tersimpan:", int(links.html_saved.sum()))

In [ ]:
# ---- RENTANG PENUH (jalankan setelah uji benar; bisa 6-11 jam, aman dicicil) ----
# Kalau terputus, jalankan ulang sel ini -> lanjut dari checkpoint.
# links, clean = run_crawl(DATE_START, DATE_END)
# print("SELESAI. link:", len(links), "| kandidat:", int(links.is_candidate.sum()))

## 8. Verifikasi output

Cek jumlah, rentang tanggal, dan buktikan mapping link <-> file HTML (dua arah).

In [ ]:
links = pd.read_parquet(LINKS_PATH)
clean = pd.read_parquet(CLEAN_PATH)
print("total link           :", len(links))
print("kandidat             :", int(links.is_candidate.sum()))
print("html tersimpan       :", int(links.html_saved.sum()))
print("artikel ter-parse    :", len(clean))
print("rentang index_date   :", links.index_date.min(), "->", links.index_date.max())
print("file .gz artikel      :", len(list((RAW/'html_article').rglob('*.html.gz'))))

# bukti mapping: dari 1 baris link -> buka file .gz -> parse balik -> cocokkan judul
r = links[links.html_saved].iloc[0]
art = parse_article(load_gz(RAW / r.html_path), r.url)
print("\nLINK :", r.url)
print("HTML :", r.html_path)
print("parse balik -> judul:", art["title"][:60], "| kata:", art["n_words"])

## Setelah selesai
Paste ringkasan sel verifikasi ke chat. Kalau cocok dengan versi lama
(~77.730 link, ~9.140 kandidat, span 2024-01-02 → 2026-09-30), lanjut ke Notebook 2 (Cleaning).